# Day 2 Project — Solution: Contact List Cleaner

A reference build of the **Load → Clean → Validate → Save** utility. This notebook is self-contained: it creates its own sample data, runs, and prints a report.

In [ ]:
import json

def clean_record(record: dict) -> dict:
    """Normalise one contact record."""
    name = ' '.join(record.get('name', '').split()).title()
    email = record.get('email', '').strip().lower()
    return {'name': name, 'email': email}


def clean_file(in_path: str, out_path: str) -> dict:
    """Load messy contacts, clean + validate, save clean, return a report."""
    # Load
    with open(in_path) as f:
        raw = json.load(f)

    # Clean (one comprehension) then Validate (keep only complete records)
    cleaned = [clean_record(r) for r in raw]
    valid = [r for r in cleaned if r['name'] and r['email']]

    # Save
    with open(out_path, 'w') as f:
        json.dump(valid, f, indent=2)

    # Report
    return {'total': len(raw), 'kept': len(valid), 'dropped': len(raw) - len(valid)}

### Create sample data and run

In [ ]:
messy = [
    {'name': '  ada  LOVELACE ', 'email': ' ADA@math.org '},
    {'name': 'alan turing',      'email': 'alan@bletchley.uk'},
    {'name': '  grace hopper',   'email': ''},
    {'email': 'anon@nowhere.io'},
    {'name': 'KATHERINE johnson','email': 'KJ@nasa.GOV '},
]
with open('contacts_raw.json', 'w') as f:
    json.dump(messy, f, indent=2)

report = clean_file('contacts_raw.json', 'contacts_clean.json')
print('Report:', report)

with open('contacts_clean.json') as f:
    print('\nClean records:')
    print(json.dumps(json.load(f), indent=2))

# tidy up the demo files
import os
for p in ('contacts_raw.json', 'contacts_clean.json'):
    if os.path.exists(p): os.remove(p)

**How it maps to the pattern**

- **Load** — `json.load` reads the raw list.
- **Clean** — `[clean_record(r) for r in raw]` normalises every record.
- **Validate** — `[r for r in cleaned if r['name'] and r['email']]` drops incomplete records.
- **Save** — `json.dump(valid, ...)` writes indented JSON.
- **Report** — counts returned as a dict.

Note how `clean_record` used `.get()` so the record missing a `name` didn't crash — it became an empty string, then got dropped at the validate step. Defensive by design.